# Modelagem probabilística e teoria de filas (Formativa04_ADS_08.pdf.ipynb)

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** SciPy, NumPy  
- **Arquivo Original:** `Formativa04_ADS_08.pdf.ipynb`  

---


# Processo de Poisson
Um processo de Poisson é modelado como uma sequência de variáveis aleatórias independentes $X_{1}, X_{2}, \cdots$ com distribuição de exponencial com parâmetro $\lambda$.Duas distribuições de probabilidade são importante no processos de Poisson: $N_{S}\left(x\right)$ e $S_{N}\left(t\right)$.<br>
$N_{S}$ é a variável aleatória que conta a quantidade de eventos que ocorrem no intervalo de tempo $S$ ($S = t_{2}-t_{1}$). Sua função de distribuição de probabilidade é dada por: $$P\left [ N_{S} = x \right ] = e^{-\lambda S}\frac{\left ( \lambda S \right )^{x}}{x!}$$
$S_{N}$ é a variável aleatória que mede o tempo decorrido até a ocorrência do próximo evento no processo de Poisson. Sua função de distribuição acumulada é dada por:$$P\left [ S_{N} \leq  x \right ] = 1 - \sum_{j=0}^{N-1}e^{-\lambda x}\frac{\left ( \lambda x \right )^{j}}{j!}$$

## Importação das bibliotecas necessárias

In [12]:
import scipy.stats as st
import time
import numpy as np

## Simulação da função de probabilidade de $N_{S}$
### Simulação interativa
A função poissonNS_I simula a função de probabilidade de $N_{S}$ com um algoritmo interativo. Recebe como argumento x, lambda, t1, t2 e nSim, e calcula a probabilidade de ocorrência de x eventos no intervalo entre t1 e t2. O argumento lbda é o parâmetro das exponenciais que definem o processo de Poisson. t1 e t2 definem o intervalo $S$. O parâmetro nSim é a quantidade de vezes que a simulação é executada.

In [13]:
def poissonNS_I(x, lbda, t1, t2, nsim):
    mu = 1/lbda
    deuCerto = 0
    for i in range(nSim):
        tempo = 0
        nEventos = 0
        while tempo <= t2:
            if (tempo >= t1):
                nEventos = nEventos + 1
            # sorteia variável exponencial e acumula em tempo
            tempo = tempo + st.expon.rvs(0, mu)
        # se quantidade de eventos = x, deuCerto
        if nEventos == x:
            deuCerto = deuCerto + 1
    return(deuCerto/nSim)

Os comandos abaixos simulam $nSim$ observações de um processos de Poisson.<br>
Imprime a proporção de vezes que o número de eventos entre t1 e t2 é igual a x (probabilidade simulada).
Imprime o valor previsto pela teoria (probabilidade teórica).
Imprime o tempo de simulação (em segundos).

In [14]:
t1 = 15
t2 = 25
S = t2-t1
lbda = 0.2
nSim = 50000
x = 5
probT = st.poisson.pmf(x, lbda*S)
inic = time.perf_counter()
probS = poissonNS_I(x, lbda, t1, t2, nSim)
fim = time.perf_counter()
print('Probabilidade simulada:  {:.4f}'.format(probS))
print('Probabilidade teórica: {:.4f}'.format(probT))
print('Tempo de simulação iterativa: {:.4f}'.format(fim-inic))

Probabilidade simulada:  0.0367
Probabilidade teórica: 0.0361
Tempo de simulação iterativa: 11.0793


### Simulação vetorial
A função poissonNS_V simula a função de probabilidade de $N_{S}$ com um algoritmo vetorial. Recebe como argumento x, lambda, t1, t2 e nSim, e calcula a probabilidade de ocorrência de x eventos no intervalo entre t1 e t2. O argumento lbda é o parâmetro das exponenciais que definem o processo de Poisson. t1 e t2 definem o intervalo $S$. O parâmetro nSim é a quantidade de vezes que a simulação é executada.<br>
No algoritmo vetorial precisamos sortear todas as exponenciais e colocar em uma matriz $X$, que terá $nSim$ linhas, cada linha contando uma observação do processo de Poisson, ou seja, os valores acumulados de tempo das variáveis aleatórias exponenciais.<br>
A maior dificuldade é determinar a quantidade de colunas da matriz $X$. O que sabemos é que deveriamos acumular uma quantidade suficiente para ultrapassar $t_{2}$. Sabemos que média dos valores dos tempos sorteados será $\mu = 1/\lambda$. Precisaremos, em média, $N= ceil\left (  t_{2}/\mu\right )$ lançamentos para chegar a $t_{2}$, onde $ceil$ é o arredondamento para cima. Um valor empírico aproximado para a quantidade de colunas necessárias é $m = ceil\left (2\cdot t_{2}/\mu\right ) $.<br>
A seguir vamos testar se esse algoritmo está correto para gerar as observações necessárias
* Sortear a matriz $X$ com $n$ linhas e $m$ colunas
* Acumular os valores das exponenciais em cada linha gerando os eventos<br>
Execute os comandos a seguir para verificar se a matriz está sendo gerada de acordo.

In [15]:
n = 5
t1 = 10
t2 = 30
lbda = 0.2
mu = 1/lbda
m = int(np.ceil(2*t2/mu))
X = np.cumsum(np.random.exponential(mu, (n, m)), 1)
print(X)
eventos = np.count_nonzero((X>t1) & (X<t2),1)

[[ 3.02818619 20.52025923 48.4423418  51.95763763 52.6084575  56.5621067
  58.04356138 63.29961607 63.71685104 64.2484613  65.4615198  75.86932139]
 [ 0.0793324   2.66229084  3.82666507  9.84685286  9.86693038 10.22304049
  20.47054996 29.388395   30.85514143 38.23128632 40.44878713 41.76584819]
 [ 3.17592643  3.92032543  4.17301827 13.60738607 16.85175208 21.89210699
  31.63073274 35.45387611 35.86650859 40.11488098 40.16293595 42.3053191 ]
 [ 1.74147269 11.44995983 11.66513752 28.84769353 28.86818309 29.59730028
  43.09395574 46.70148848 49.49983804 49.61519088 51.55376181 56.01385611]
 [14.50844843 17.69528674 22.2323186  28.89359989 29.30465986 48.27758528
  53.24641988 61.78387348 61.89374884 66.71988772 68.03458462 71.07851668]]


Para implementar a função poissonNS_V, será nenessário usar os passos anteriores mais dois passos:
* Contar quantos eventos ocorreram entre t1 e t1.
* Contar a quantidade de linhas onde quantidade de eventos contados entre t1 e t2 é igual a x.

In [16]:
def poissonNS_V(x, lbda, t1, t2, nSim):
    # coloque seu código aqui
    # Calcula mu
    mu = 1/lbda

    # calcula m
    m = int(np.ceil(2*t2/mu))

    # Sorteia matriz X
    X = np.cumsum(np.random.exponential(mu, (nSim,m)), 1)

    # Calcula matriz eventos
    eventos = np.count_nonzero((X>t1) & (X<t2),1)

    # Calcula probabilidade
    prob = np.count_nonzero(eventos == x)/nSim
    # Alterar o comando abaixo
    return prob

## Testar a simulação
Se o seu código estiver correto, os valores da probabilidade teórica e simulada devem ser muito próximos.

In [17]:
t1 = 15
t2 = 25
S = t2-t1
lbda = 0.2
nSim = 100000
x=3
probT = st.poisson.pmf(x, lbda*S)
inic = time.perf_counter()
probS = poissonNS_V(x, lbda, t1, t2, nSim)
fim = time.perf_counter()
print('Probabilidade simulada:  {:.4f}'.format(probS))
print('Probabilidade teórica: {:.4f}'.format(probT))
print('Tempo de simulação vetorial: {:.4f}'.format(fim-inic))

Probabilidade simulada:  0.1815
Probabilidade teórica: 0.1804
Tempo de simulação vetorial: 0.0383


## Entrega
Exportar para pdf.<br>
Renomear para Formativa05-nome-do-grupo.pdf<br>
Fazer upload no AVA.